# AI Escape Room — Setup

Run these cells from top to bottom. In Google Colab, add your Gemini API key in **Secrets** using the name `GEMINI_API_KEY`, and enable notebook access to that secret.

In [1]:
!pip install -q "google-genai>=2.12.1,<3" gradio

In [2]:
import google.genai
import google.auth
import gradio

print("Gemini SDK:", google.genai.__version__)
print("Google Auth:", google.auth.__version__)
print("Gradio:", gradio.__version__)

Gemini SDK: 2.12.1
Google Auth: 2.49.0
Gradio: 6.31.0


In [3]:
from google.colab import userdata
from google import genai

# Retrieve API key from Colab Secrets
api_key = userdata.get("GEMINI_API_KEY")

# Connect to Gemini
client = genai.Client(api_key=api_key)

print("Gemini API connected successfully!")

Gemini API connected successfully!


# Version 2 — Python-managed escape room

**This is the start of Version 2.** Python stores the authoritative game state, validates puzzle answers in order, controls drawer/key/exit progression, and Gemini generates the narrative based on verified results.

In [4]:
game_master_prompt = """
You are the Game Master of an interactive AI escape room called
'The Forgotten Library'.

YOUR ROLE:
- Act as an immersive, mysterious and friendly Game Master.
- Describe the library and its objects.
- Respond naturally to the player's actions.
- Give hints when the player is genuinely stuck.
- Maintain continuity using the conversation history.

GAME ENVIRONMENT:

The player wakes up inside a locked library.

There are exactly FIVE interactive objects:

1. Bookshelf
2. Painting
3. Desk
4. Locked Drawer
5. Exit Door

PUZZLES:

Puzzle 1 - Bookshelf:
The player discovers unusual books containing hidden numbers.

Puzzle 2 - Painting:
The painting contains a hidden message that helps interpret the numbers.

Puzzle 3 - Desk:
A handwritten note provides the final clue needed to determine the four-digit combination.

FINAL ESCAPE:
The correct combination is 2749.
The locked drawer contains the key to the exit door.
The player must investigate the clues, unlock the drawer,
collect the key and use it to escape.

GAME RULES:

- Only describe objects that exist in the library.
- Do not introduce additional interactive objects.
- Encourage exploration.
- Do not immediately reveal all puzzle answers.
- Give clues progressively.
- Remember the player's previous messages.
- Do not declare the player escaped before the final challenge.
- If the player asks for a hint, provide a useful but gradual hint.
- Keep responses engaging and relatively short.

IMPORTANT:
This is Version 2. Python is responsible for validating game actions
and maintaining the authoritative game state. You provide the narrative
and must follow the verified results supplied by Python.

Begin by welcoming the player and describing the library.
Ask what they would like to investigate first.
"""

## 1. Python game-state manager

In [5]:

# ==========================================
# AI ESCAPE ROOM - GAME STATE MANAGER
# ==========================================

def create_game_state():

    return {
        "inventory": [],

        "puzzles_solved": [],

        "current_room": "The Forgotten Library",

        "hints_used": 0,

        "game_status": "In Progress",

        "drawer_unlocked": False,

        "exit_unlocked": False
    }


# ==========================================
# FUNCTION 1: COLLECT OBJECTS
# ==========================================

def collect_item(state, item):

    if item not in state["inventory"]:
        state["inventory"].append(item)
        return f"{item} added to inventory!"

    return f"You already have {item}."


# ==========================================
# FUNCTION 2: SOLVE PUZZLES
# ==========================================

def solve_puzzle(state, puzzle_name):

    if puzzle_name not in state["puzzles_solved"]:
        state["puzzles_solved"].append(puzzle_name)
        return f"{puzzle_name} solved!"

    return f"{puzzle_name} is already solved."


# ==========================================
# FUNCTION 3: UNLOCK DRAWER
# ==========================================

def unlock_drawer(state, combination):

    if state["drawer_unlocked"]:
        return "The drawer is already unlocked."

    if combination == "2749":

        state["drawer_unlocked"] = True

        solve_puzzle(state, "Final Combination")

        return "Correct! The drawer is unlocked. You found the key!"

    return "Incorrect combination. Keep investigating the clues."


# ==========================================
# FUNCTION 4: USE HINT
# ==========================================

def use_hint(state):

    state["hints_used"] += 1

    return f"Hint used! Total hints: {state['hints_used']}"


# ==========================================
# FUNCTION 5: ESCAPE THE ROOM
# ==========================================

def escape_room(state):

    if not state["drawer_unlocked"]:
        return "You must unlock the drawer first!"

    if "Old Key" not in state["inventory"]:
        return "You need to collect the Old Key first!"

    state["exit_unlocked"] = True
    state["game_status"] = "Completed"

    return "Congratulations! You escaped The Forgotten Library!"


# ==========================================
# FUNCTION 6: DISPLAY GAME STATUS
# ==========================================

def display_game_status(state):

    print("========== GAME STATUS ==========")

    print("Room:", state["current_room"])

    print("Inventory:", state["inventory"])

    print("Puzzles Solved:",
          state["puzzles_solved"])

    print("Hints Used:", state["hints_used"])

    print("Drawer Unlocked:",
          state["drawer_unlocked"])

    print("Exit Unlocked:",
          state["exit_unlocked"])

    print("Game Status:", state["game_status"])

    print("=================================")


print("Game State Manager initialized successfully!")


Game State Manager initialized successfully!


## 2. Drawer validation rule

In [6]:
def unlock_drawer(state, combination):

    if state["drawer_unlocked"]:
        return "The drawer is already unlocked."

    required_puzzles = ["Bookshelf", "Painting", "Desk"]

    if not all(puzzle in state["puzzles_solved"]
               for puzzle in required_puzzles):
        return "You must solve all three puzzles before unlocking the drawer!"

    if str(combination).strip() == "2749":
        state["drawer_unlocked"] = True
        return "Correct combination! The drawer is unlocked. You can now collect the Old Key."

    return "Incorrect combination. Keep investigating the clues."

## 3. Puzzle answer validation

Correct puzzle answers: Bookshelf = `2`, Painting = `7`, Desk = `49`. They must be solved in that order.

In [7]:
def check_puzzle_answer(state, puzzle, answer):

    answers = {
        "Bookshelf": "2",
        "Painting": "7",
        "Desk": "49"
    }

    if puzzle not in answers:
        return "Unknown puzzle."

    if puzzle in state["puzzles_solved"]:
        return "You have already solved this puzzle!"

    # Require puzzles to be solved in order
    order = ["Bookshelf", "Painting", "Desk"]
    next_puzzle = order[len(state["puzzles_solved"])]

    if puzzle != next_puzzle:
        return f"Solve the {next_puzzle} puzzle first."

    if str(answer).strip() == answers[puzzle]:
        solve_puzzle(state, puzzle)
        return f"Correct! You solved the {puzzle} puzzle."

    return "Incorrect answer. Look for more clues!"

## 4. Gemini action recognizer

Gemini translates the player's message into a structured action. Python still validates whether that action is allowed.

In [8]:

import re
import json

def recognize_player_action(player_message, chat=None):

    message = player_message.lower().strip()

    puzzles = {
        "bookshelf": "Bookshelf",
        "painting": "Painting",
        "desk": "Desk"
    }

    # Detect puzzle answers
    for keyword, puzzle_name in puzzles.items():

        if keyword in message:

            numbers = re.findall(r"\b\d+\b", message)

            if numbers:
                return {
                    "action": "submit_answer",
                    "puzzle": puzzle_name,
                    "answer": numbers[-1],
                    "item": "",
                    "combination": ""
                }

    # Detect drawer combination
    if any(word in message for word in
           ["drawer", "combination", "unlock"]):

        numbers = re.findall(r"\b\d{4}\b", message)

        if numbers:
            return {
                "action": "unlock_drawer",
                "puzzle": "",
                "answer": "",
                "item": "",
                "combination": numbers[-1]
            }

    # Detect collecting the key
    if any(word in message for word in
           ["collect", "take", "pick up"]) and "key" in message:

        return {
            "action": "collect_item",
            "puzzle": "",
            "answer": "",
            "item": "Old Key",
            "combination": ""
        }

    # Detect hints
    if "hint" in message:

        return {
            "action": "use_hint",
            "puzzle": "",
            "answer": "",
            "item": "",
            "combination": ""
        }

    # Detect escape
    if "escape" in message or "leave the library" in message:

        return {
            "action": "escape",
            "puzzle": "",
            "answer": "",
            "item": "",
            "combination": ""
        }

    return {
        "action": "explore",
        "puzzle": "",
        "answer": "",
        "item": "",
        "combination": ""
    }


## 5. Python action validator

In [9]:

def process_player_action(state, action_result):

    action = action_result.get("action", "unknown")

    if action == "submit_answer":

        puzzle = action_result.get("puzzle", "")
        answer = action_result.get("answer", "")

        return check_puzzle_answer(state, puzzle, answer)

    elif action == "collect_item":

        item = action_result.get("item", "")

        if item != "Old Key":
            return "That item cannot be collected."

        if not state["drawer_unlocked"]:
            return "You must unlock the drawer before collecting the Old Key."

        return collect_item(state, item)

    elif action == "unlock_drawer":

        combination = action_result.get("combination", "")
        return unlock_drawer(state, combination)

    elif action == "use_hint":

        return get_game_hint(state)

    elif action == "escape":

        return escape_room(state)

    elif action == "explore":

        return "Explore the library and look for clues."

    return "I couldn't identify that action."


## 6. Game progress display

In [10]:
def format_game_status(state):

    puzzles = ["Bookshelf", "Painting", "Desk"]

    completed = len([
        puzzle for puzzle in puzzles
        if puzzle in state["puzzles_solved"]
    ])

    inventory = ", ".join(state["inventory"]) if state["inventory"] else "Empty"

    puzzle_status = "\n".join([
        f"{'✅' if puzzle in state['puzzles_solved'] else '🔒'} {puzzle}"
        for puzzle in puzzles
    ])

    return f"""
## 📚 Game Progress

**Location:** {state['current_room']}

### 🧩 Puzzle Progress
{completed}/3 completed

{puzzle_status}

### 🎒 Inventory
{inventory}

### 🔐 Locked Drawer
{"✅ Unlocked" if state['drawer_unlocked'] else "🔒 Locked"}

### 🚪 Exit Door
{"✅ Unlocked" if state['exit_unlocked'] else "🔒 Locked"}

### 💡 Hints Used
{state['hints_used']}

### 🎮 Game Status
{state['game_status']}
"""

## 7. Hint system

In [11]:
def get_game_hint(state):

    hints = {
        "Bookshelf": [
            "Look carefully at the unusual books.",
            "Some books contain hidden numbers.",
            "Focus on the number 2."
        ],
        "Painting": [
            "Examine the details in the painting.",
            "Look for a hidden message.",
            "The clue points towards the number 7."
        ],
        "Desk": [
            "Search the handwritten note carefully.",
            "The note contains the final numerical clue.",
            "The answer is 49."
        ]
    }

    order = ["Bookshelf", "Painting", "Desk"]

    for puzzle in order:

        if puzzle not in state["puzzles_solved"]:

            hint_index = min(state["hints_used"], 2)

            state["hints_used"] += 1

            return f"💡 Hint for {puzzle}: {hints[puzzle][hint_index]}"

    if not state["drawer_unlocked"]:
        state["hints_used"] += 1
        return "💡 Combine the clues you discovered to unlock the drawer."

    return "💡 You have solved all the puzzles! Collect the Old Key and find your way out."

## 8. Story context based on verified state

In [12]:

def get_story_context(state):

    solved = state["puzzles_solved"]

    if state["game_status"] == "Completed":
        return """
        FINAL SCENE:
        The player has escaped The Forgotten Library.
        Describe the library doors opening, the atmosphere changing,
        and give the player a satisfying ending.
        """

    if state["exit_unlocked"]:
        return """
        FINAL CHALLENGE:
        The exit is unlocked.
        Describe the mysterious door and encourage the player to escape.
        """

    if state["drawer_unlocked"]:
        return """
        NEW STORY STAGE:
        The locked drawer has opened.
        The Old Key is inside.
        Let the player collect it and investigate the exit door.
        """

    if len(solved) == 3:
        return """
        FINAL PUZZLE STAGE:
        All three puzzles are solved.
        The clues reveal the combination 2749.
        Encourage the player to enter it into the locked drawer.
        """

    if len(solved) == 2:
        return """
        STORY STAGE:
        Two puzzles have been solved.
        The library is becoming strangely quiet.
        Encourage the player to investigate the final puzzle.
        """

    if len(solved) == 1:
        return """
        STORY STAGE:
        The first puzzle has been solved.
        A mysterious sound comes from deeper inside the library.
        Encourage further exploration.
        """

    return """
    INTRODUCTION:
    The player is trapped inside The Forgotten Library.
    Create a mysterious atmosphere.
    Encourage the player to investigate the Bookshelf, Painting and Desk.
    """


## 9. Updated Game Master rules

In [13]:

game_master_prompt += """

PUZZLE DESIGN AND ANSWERS

There are exactly three story puzzles.

PUZZLE 1 — BOOKSHELF
- The player must investigate unusual books.
- A hidden marking reveals the number 2.
- The correct answer is 2.
- Give subtle clues before revealing the answer.

PUZZLE 2 — PAINTING
- The player must inspect the mysterious painting.
- A hidden symbol or message reveals the number 7.
- The correct answer is 7.
- Encourage the player to examine the painting carefully.

PUZZLE 3 — DESK
- The player must investigate a handwritten note.
- The note reveals the number 49.
- The correct answer is 49.
- Encourage the player to connect the previous clues.

FINAL COMBINATION
- The three puzzle answers combine in order:
  2 + 7 + 49 = 2749.
- The correct drawer combination is 2749.
- The drawer contains the Old Key.
- The player must collect the Old Key after opening the drawer.
- The Old Key is required to escape.

IMPORTANT GAME RULES
- Never claim a puzzle is solved unless the Python game system confirms it.
- Never claim an item was collected unless Python confirms it.
- Never unlock the drawer without Python confirmation.
- Do not reveal all puzzle answers at once.
- Encourage exploration and logical thinking.
- Maintain a mysterious and immersive atmosphere.
"""

print("Game Master instructions updated successfully!")


Game Master instructions updated successfully!


## 10. Gradio interface — Version 2

This interface calls the Python action validator before asking Gemini to narrate the result. Restart resets both the conversation and Python game state.

In [ ]:

import gradio as gr

def play_game(message, history, chat, state):

    if not message.strip():
        return history, chat, state, format_game_status(state)

    try:
        # Create Gemini conversation if needed
        if chat is None:
            chat = client.chats.create(
                model="gemini-3.5-flash-lite",
                config={
                    "system_instruction": game_master_prompt,
                    "temperature": 0.7
                }
            )

        # Recognize player action directly using Python
        action_result = recognize_player_action(message)

        # Validate action using Python
        action_feedback = process_player_action(
            state,
            action_result
        )

        # Get updated story context
        story_context = get_story_context(state)

        # Send verified information to Gemini
        game_message = f"""
PLAYER MESSAGE:
{message}

PYTHON GAME SYSTEM RESULT:
{action_feedback}

VERIFIED GAME STATE:
{format_game_status(state)}

CURRENT STORY:
{story_context}

Respond naturally as the Game Master.
Respect the verified game state.
Never claim an action succeeded if Python rejected it.
"""

        response = chat.send_message(game_message)

        bot_reply = response.text

        history = history or []

        history = history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": bot_reply}
        ]

        return history, chat, state, format_game_status(state)

    except Exception as e:

        history = history or []

        history = history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": f"Error: {str(e)}"}
        ]

        return history, chat, state, format_game_status(state)


def restart_game():

    new_state = create_game_state()

    return [], None, new_state, format_game_status(new_state)


with gr.Blocks(title="AI Escape Room") as demo:

    gr.Markdown("""
    # 🔐 The Forgotten Library

    Welcome, Explorer!

    You are trapped inside a mysterious library.

    Investigate objects, solve puzzles and discover
    the secret that will help you escape.

    **Your mission: Escape the library!**
    """)

    with gr.Row():

        with gr.Column(scale=2):

            chatbot = gr.Chatbot(
                label="Escape Room Adventure",
                height=450
            )

            message_box = gr.Textbox(
                placeholder="What would you like to investigate?",
                label="Your message"
            )

            with gr.Row():

                send_button = gr.Button(
                    "Send Message",
                    variant="primary"
                )

                restart_button = gr.Button("Restart Game")

        with gr.Column(scale=1):

            status = gr.Markdown(
                format_game_status(create_game_state())
            )

    chat_state = gr.State(None)
    game_state = gr.State(create_game_state())

    send_button.click(
        fn=play_game,
        inputs=[message_box, chatbot, chat_state, game_state],
        outputs=[chatbot, chat_state, game_state, status]
    ).then(
        fn=lambda: "",
        outputs=message_box
    )

    message_box.submit(
        fn=play_game,
        inputs=[message_box, chatbot, chat_state, game_state],
        outputs=[chatbot, chat_state, game_state, status]
    ).then(
        fn=lambda: "",
        outputs=message_box
    )

    restart_button.click(
        fn=restart_game,
        outputs=[chatbot, chat_state, game_state, status]
    )


demo.queue()

demo.launch(
    share=True,
    debug=True,
    theme=gr.themes.Soft()
)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9eac2ea82d3fa0bfa6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 11. Automated end-to-end tests

Run these tests in a separate test state; they do not change the live Gradio session.

In [15]:

# Create a separate test game
test_state = create_game_state()

print("TEST 1: Try solving puzzles out of order")
result = check_puzzle_answer(test_state, "Painting", "7")
print(result)

print("\nTEST 2: Submit an incorrect answer")
result = check_puzzle_answer(test_state, "Bookshelf", "5")
print(result)

print("\nTEST 3: Solve all three puzzles")

for puzzle, answer in [
    ("Bookshelf", "2"),
    ("Painting", "7"),
    ("Desk", "49")
]:
    result = check_puzzle_answer(test_state, puzzle, answer)
    print(result)

print("\nTEST 4: Unlock the drawer")
print(unlock_drawer(test_state, "2749"))

print("\nTEST 5: Collect the Old Key")
print(collect_item(test_state, "Old Key"))

print("\nTEST 6: Escape the library")
print(escape_room(test_state))

print("\nFINAL GAME STATUS")
display_game_status(test_state)

# Verify the complete journey
assert len(test_state["puzzles_solved"]) == 3
assert test_state["drawer_unlocked"] is True
assert "Old Key" in test_state["inventory"]
assert test_state["game_status"] == "Completed"

print("\nALL TESTS PASSED!")


TEST 1: Try solving puzzles out of order
Solve the Bookshelf puzzle first.

TEST 2: Submit an incorrect answer
Incorrect answer. Look for more clues!

TEST 3: Solve all three puzzles
Correct! You solved the Bookshelf puzzle.
Correct! You solved the Painting puzzle.
Correct! You solved the Desk puzzle.

TEST 4: Unlock the drawer
Correct combination! The drawer is unlocked. You can now collect the Old Key.

TEST 5: Collect the Old Key
Old Key added to inventory!

TEST 6: Escape the library
Congratulations! You escaped The Forgotten Library!

FINAL GAME STATUS
========== GAME STATUS ==========
Room: The Forgotten Library
Inventory: ['Old Key']
Puzzles Solved: ['Bookshelf', 'Painting', 'Desk']
Hints Used: 0
Drawer Unlocked: True
Exit Unlocked: True
Game Status: Completed

ALL TESTS PASSED!


## 12. Adversarial/security tests

In [16]:

# Version 2 adversarial testing

def run_security_tests():

    results = []

    # Test 1: Claim puzzle solved without correct answer
    state = create_game_state()
    result = check_puzzle_answer(state, "Bookshelf", "wrong")
    results.append(("Incorrect puzzle answer", result,
                    "Bookshelf" not in state["puzzles_solved"]))

    # Test 2: Incorrect drawer code
    state = create_game_state()
    for puzzle, answer in [
        ("Bookshelf", "2"),
        ("Painting", "7"),
        ("Desk", "49")
    ]:
        check_puzzle_answer(state, puzzle, answer)

    result = unlock_drawer(state, "1234")
    results.append(("Incorrect drawer code", result,
                    state["drawer_unlocked"] is False))

    # Test 3: Collect key before unlocking drawer
    state = create_game_state()
    result = collect_item(state, "Old Key")
    results.append(("Collect key before unlocking", result,
                    "Old Key" not in state["inventory"]))

    # Test 4: Administrator bypass attempt
    state = create_game_state()
    result = escape_room(state)
    results.append(("Administrator bypass attempt", result,
                    state["game_status"] != "Completed"))

    # Test 5: Attempt to unlock drawer without puzzles
    state = create_game_state()
    result = unlock_drawer(state, "2749")
    results.append(("Skip all puzzles", result,
                    state["drawer_unlocked"] is False))

    print("VERSION 2 SECURITY TEST RESULTS")
    print("=" * 60)

    for name, feedback, passed in results:
        print(f"\nTest: {name}")
        print(f"Result: {feedback}")
        print("Status:", "PASS" if passed else "FAIL")

    print("\nTotal passed:",
          sum(test[2] for test in results), "/ 5")


run_security_tests()


VERSION 2 SECURITY TEST RESULTS

Test: Incorrect puzzle answer
Result: Incorrect answer. Look for more clues!
Status: PASS

Test: Incorrect drawer code
Result: Incorrect combination. Keep investigating the clues.
Status: PASS

Test: Collect key before unlocking
Result: Old Key added to inventory!
Status: FAIL

Test: Administrator bypass attempt
Result: You must unlock the drawer first!
Status: PASS

Test: Skip all puzzles
Result: You must solve all three puzzles before unlocking the drawer!
Status: PASS

Total passed: 4 / 5


**Test puzzle registration**

In [17]:

test_state = create_game_state()

print(process_player_action(
    test_state,
    recognize_player_action("Bookshelf answer is 2")
))

print(process_player_action(
    test_state,
    recognize_player_action("Painting answer is 7")
))

print(process_player_action(
    test_state,
    recognize_player_action("Desk answer is 49")
))

print("\nFINAL PUZZLE STATUS:")
print(test_state["puzzles_solved"])


Correct! You solved the Bookshelf puzzle.
Correct! You solved the Painting puzzle.
Correct! You solved the Desk puzzle.

FINAL PUZZLE STATUS:
['Bookshelf', 'Painting', 'Desk']


**Test the complete puzzle sequence**

In [18]:
test_state = create_game_state()

messages = [
    "Bookshelf answer is 2",
    "Painting answer is 7",
    "Desk answer is 49",
    "Unlock drawer with 2749",
    "Collect the Old Key",
    "Escape the library"
]

for message in messages:

    action = recognize_player_action(message)

    result = process_player_action(test_state, action)

    print(f"\nPlayer: {message}")
    print(f"Game: {result}")

print("\nFINAL STATUS:")
print(format_game_status(test_state))


Player: Bookshelf answer is 2
Game: Correct! You solved the Bookshelf puzzle.

Player: Painting answer is 7
Game: Correct! You solved the Painting puzzle.

Player: Desk answer is 49
Game: Correct! You solved the Desk puzzle.

Player: Unlock drawer with 2749
Game: Correct combination! The drawer is unlocked. You can now collect the Old Key.

Player: Collect the Old Key
Game: Old Key added to inventory!

Player: Escape the library
Game: Congratulations! You escaped The Forgotten Library!

FINAL STATUS:

## 📚 Game Progress

**Location:** The Forgotten Library

### 🧩 Puzzle Progress
3/3 completed

✅ Bookshelf
✅ Painting
✅ Desk

### 🎒 Inventory
Old Key

### 🔐 Locked Drawer
✅ Unlocked

### 🚪 Exit Door
✅ Unlocked

### 💡 Hints Used
0

### 🎮 Game Status
Completed

